# Content Bases Filtering

## Import Library

In [1]:
import pandas as pd;

## Load Data

In [2]:
#Loading Dataset
animes = pd.read_csv("data/animes.csv", encoding="ISO-8859-1")
animes.head() #display in table

# print(animes)

,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,link
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...","Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,https://myanimelist.net/anime/28891/Haikyuu_Se...
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,"['Drama', 'Music', 'Romance', 'School', 'Shoun...","Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,https://myanimelist.net/anime/23273/Shigatsu_w...
2,34599,Made in Abyss,The Abyssâa gaping chasm stretching down int...,"['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...","Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,https://myanimelist.net/anime/34599/Made_in_Abyss
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...","['Action', 'Military', 'Adventure', 'Comedy', ...","Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,https://myanimelist.net/anime/5114/Fullmetal_A...
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,"['Action', 'Mystery', 'Supernatural', 'Vampire']","Jan 6, 2017",1.0,214621,502,22.0,8.83,https://myanimelist.net/anime/31758/Kizumonoga...


## Clean Data

In [3]:
#for checking is there any null value
print("Total null value in dataset:")
print(animes.isnull().sum())



Total null value in dataset:
uid             0
title           0
synopsis      546
genre           0
aired           0
episodes      207
members         0
popularity      0
ranked         62
score           0
link            0
dtype: int64


In [4]:
#cleaning null value
animes['synopsis'] = animes['synopsis'].fillna("")
animes['genre'] = animes['genre'].fillna("")

#for checking is there still have any null value
print("\nTotal null value in dataset (specific column):")
print(animes[['uid','title','synopsis', 'genre']].isnull().sum())



Total null value in dataset (specific column):
uid         0
title       0
synopsis    0
genre       0
dtype: int64


In [5]:
# Keep only useful columns
animes = animes[['uid', 'title', 'synopsis', 'genre']]

## NLP - TF-IDF

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine synopsis + genres into one content field
animes['content'] = animes['synopsis'] + " " + animes['genre'].apply(lambda x: " ".join(eval(x)) if isinstance(x, str) else "")

#Remove all english stop words such as 'the', 'a'
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(animes['content'])
print(tfidf_matrix)


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 543319 stored elements and shape (15508, 42773)>
  Coords	Values
  (0, 12720)	0.07188030140358712
  (0, 27818)	0.11781509973062658
  (0, 17849)	0.10668574332236502
  (0, 15997)	0.15193097746938503
  (0, 19412)	0.36854908072480974
  (0, 32841)	0.07806787236398857
  (0, 40623)	0.32855484494069437
  (0, 37370)	0.1219916544064078
  (0, 3079)	0.07741411507726825
  (0, 30584)	0.1348096045939041
  (0, 10600)	0.08633765237258405
  (0, 1410)	0.0953824788896878
  (0, 7331)	0.08695820751477708
  (0, 35470)	0.0838402124225546
  (0, 38541)	0.08160992618117491
  (0, 17777)	0.07307385904617315
  (0, 30403)	0.08618668455814867
  (0, 18055)	0.10096245893680833
  (0, 21943)	0.0612245238438107
  (0, 35598)	0.08918706500408344
  (0, 31581)	0.07662972684000174
  (0, 25632)	0.1348096045939041
  (0, 1347)	0.09300230925773875
  (0, 21271)	0.07686050074476028
  (0, 38644)	0.07307385904617315
  :	:
  (15507, 39707)	0.08797635900971588
  (15507, 28699

## Function

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_by_query(query, top_n=20):
    # Transform input into a TF-IDF vector
    query_vec = vectorizer.transform([query])

    sim_scores = cosine_similarity(query_vec, tfidf_matrix).ravel()

    # Get top_n highest similarity scores, sorted in descending order
    top_indices = sim_scores.argsort()[-top_n:][::-1]
    
    # Return similar anime result
    return animes['title'].iloc[top_indices].tolist()

# Example
print(recommend_by_query("comedy and school"))
print(recommend_by_query("Is there any sports high school anime recommended?"))


['Yuyushiki: Komarasetari, Komarasaretari', 'Tales of HR', '3-Nen D-Gumi Glass no Kamen: Tobidase! Watashitachi no Victory Road', 'Wake Up, Girl Zoo! Miyagi PR de Go!', 'Kana Kana Kazoku x Himitsukessha Taka no Tsume Collaboration Film', 'Shiodome Cable TV', 'Sushi Azarashi', 'Sugai-kun to Kazoku Ishi', 'Mandamgangho', 'Umeboshi Denka: Uchuu no Hate kara Panparopan!', 'Yonimo Kimyou na Manâ\x98\x86Gatarou', 'Yuuyake Dandan: Manner Movies', 'Hentatsu', 'Ton-Ton Atta to Niigata no Mukashibanashi', 'Chamebou Kuukijuu no Maki', 'Catchy-kun no Nice Catch!', 'AAA de Ikou!!: Yuuna & Akiko', 'Bakabon Osomatsu no Karee wo Tazunete Sansenri', 'Dekobou Shingachou: Meian no Shippai', 'Chuunibyou demo Koi ga Shitai! Movie: Take On Me Mini-Drama - Koukai Chokuzen no... Kaiko Gekijou']
['CMFU Xueyuan: Shenshi Ji Jijian', 'CMFU Xueyuan: Wangzi Peng Peng Qiu', 'Nae Ireumeun Dokgotak', 'Ashita Tenki ni Naare Omake', 'Dokgotak: Taeyang-eul Hyanghae Deonjyeola', 'Kagayake! Yuujou no V Sign', 'Girls Jockey

## Setup
- Data Cleaning process

In [ ]:
import pandas as pd;
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#Loading Dataset
animes = pd.read_csv("data/animes.csv", encoding="ISO-8859-1")
animes.head() #display in table

#for checking is there any null value
print("Total null value in dataset:")
print(animes.isnull().sum())

#cleaning null value
animes['synopsis'] = animes['synopsis'].fillna("")
animes['genre'] = animes['genre'].fillna("")

#for checking is there still have any null value
print("\nTotal null value in dataset (specific column):")
print(animes[['uid','title','synopsis', 'genre']].isnull().sum())

# Keep only useful columns
animes = animes[['uid', 'title', 'synopsis', 'genre']]

# Combine synopsis + genres into one content field
animes['content'] = animes['synopsis'] + " " + animes['genre'].apply(lambda x: " ".join(eval(x)) if isinstance(x, str) else "")

## NLP Processing

In [ ]:
#Remove all english stop words such as 'the', 'a'
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(animes['content'])
print(tfidf_matrix)

## Function
- Content-Based Filtering

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_by_query(query, top_n=20):
    query_vec = vectorizer.transform([query])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).ravel()
    top_indices = sim_scores.argsort()[-top_n:][::-1]
    return animes['title'].iloc[top_indices].tolist()

## Sample Test
- Test Run Content-Based Filtering

In [ ]:
# Example
print(recommend_by_query("comedy and school")) 
print(recommend_by_query("Is there any sports high school anime recommended?"))